In [2]:
import pandas as pd
import numpy as np
from collections import Counter

# FUNCȚIE PENTRU GENERARE CORPUS SEMANTIC (SFE-5)

def generate_corpus(file_path):
    
    df = pd.read_csv(file_path)
    X = df.drop("Label", axis=1)

    # Quintile pentru cele 5 caracteristici de bază
    quintiles = {
        "bytes": X["Flow Bytes/s"].quantile([0.2, 0.4, 0.6, 0.8]).values,
        "packets": X["Flow Packets/s"].quantile([0.2, 0.4, 0.6, 0.8]).values,
        "duration": X["Flow Duration"].quantile([0.2, 0.4, 0.6, 0.8]).values,
        "pkt_std": X["Packet Length Std"].quantile([0.2, 0.4, 0.6, 0.8]).values,
        "ratio": X["Down/Up Ratio"].quantile([0.2, 0.4, 0.6, 0.8]).values
    }

    def categorize(value, thresholds):
        if value < thresholds[0]:
            return "very low"
        elif value < thresholds[1]:
            return "low"
        elif value < thresholds[2]:
            return "moderate"
        elif value < thresholds[3]:
            return "high"
        else:
            return "very high"

    def semantic_encode(row):
        duration = categorize(row["Flow Duration"], quintiles["duration"])
        throughput = categorize(row["Flow Bytes/s"], quintiles["bytes"])
        packet_rate = categorize(row["Flow Packets/s"], quintiles["packets"])
        variability = categorize(row["Packet Length Std"], quintiles["pkt_std"])
        directionality = categorize(row["Down/Up Ratio"], quintiles["ratio"])
        protocol = "TCP" if row["Protocol"] == 6 else "UDP"

        return (
            f"A {duration} {protocol}-based flow with {throughput} throughput, "
            f"{packet_rate} packet rate, {directionality} directional balance, "
            f"{variability} packet size variability."
        )

    print(f"Generare corpus pentru {file_path} ...")
    corpus = [semantic_encode(X.iloc[i]) for i in range(len(X))]

    return set(corpus)

# GENERARE SETURI DE TIPARE

benign_patterns = generate_corpus("Benign_All_Flow_Clean_FINAL.csv")
ddos_patterns = generate_corpus("DDoS_Cleaned.csv")

# CALCUL INTERSECȚIE

common_patterns = benign_patterns.intersection(ddos_patterns)

print("ANALIZĂ SUPORT COMUN")
print("Tipare Benign:", len(benign_patterns))
print("Tipare DDoS:", len(ddos_patterns))
print("Tipare comune:", len(common_patterns))

print("Overlap relativ în Benign:", 
      round(len(common_patterns) / len(benign_patterns), 6))

print("Overlap relativ în DDoS:", 
      round(len(common_patterns) / len(ddos_patterns), 6))

print("==================================================")


Generare corpus pentru Benign_All_Flow_Clean_FINAL.csv ...
Generare corpus pentru DDoS_Cleaned.csv ...
ANALIZĂ SUPORT COMUN
Tipare Benign: 506
Tipare DDoS: 263
Tipare comune: 63
Overlap relativ în Benign: 0.124506
Overlap relativ în DDoS: 0.239544
